In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from pyspark.sql.functions import explode, lower, col, collect_set
import pandas as pd

spark = SparkSession.builder.appName('IngredientFrequency').getOrCreate()

# Strict schema
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])

df = spark.read.schema(ingredients_schema).parquet(
    '../output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet/part-00000-33e03f78-d7e8-41fa-a09f-4a8c4605dc35-c000.snappy.parquet'
)

In [30]:
# Explode ingredients and normalize
from pyspark.sql.functions import trim, regexp_replace
pattern = r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]' #pattern
exploded = df.select('fdc_id', 'description', explode('all_ingredients').alias('ingredient'))
exploded = exploded.withColumn('ingredient_norm', trim(lower(regexp_replace(col('ingredient'), pattern, ''))))
# Remove duplicates: ingredient counted once per food
unique_ingredients = exploded.groupBy('fdc_id', 'description').agg(collect_set('ingredient_norm').alias('ingredients_set'))
exploded_unique = unique_ingredients.select('fdc_id', 'description', explode('ingredients_set').alias('ingredient_norm'))

In [ ]:
from pyspark.sql.functions import count
import os
os.makedirs('../output/ingredient_frequency', exist_ok=True)
ingredient_counts = exploded_unique.groupBy('ingredient_norm').agg(
    count('fdc_id').alias('food_count'),
    collect_set('description').alias('examples')
)
ingredient_counts = ingredient_counts.orderBy(col('food_count').desc())
top10 = ingredient_counts.limit(10).toPandas()
rare10 = ingredient_counts.orderBy(col('food_count').asc()).limit(10).toPandas()

# Save all ingredient frequencies to CSV
ingredient_counts.toPandas().to_csv('../output/ingredient_frequency/ingredient_frequency.csv', index=False)

In [32]:
display(top10[['ingredient_norm', 'food_count', 'examples']])

,ingredient_norm,food_count,examples
0,salt,2514,"[LEMON RAPID HYDRATION MIX, LEMON, SPICY ARRAB..."
1,sugar,1831,"[SMOKED SAUSAGE WITH GARLIC, GARLIC, CHOCOLATE..."
2,water,1765,"[SMOKED SAUSAGE WITH GARLIC, GARLIC, KEY FOOD,..."
3,citric acid,1070,"[SOUR CANDY CANES, SOUR, LEMON RAPID HYDRATION..."
4,folic acid,813,"[CPB OTG CKN STARS, CHOCOLATE CHIP COOKIES, CH..."
5,natural flavor,761,"[40/3.2OZ STRAWBERRY APPLE SAUCE POUCH CASE, O..."
6,niacin,704,"[ORGANIC CRACKERS, PAN DE MAIZ CORN BREAD, PAN..."
7,riboflavin,667,"[ORGANIC CRACKERS, CPB OTG CKN STARS, CHOCOLAT..."
8,corn syrup,600,"[CEREAL WITH NUTTY PECAN BUNCHES, Black Oak Sm..."
9,soy lecithin,596,"[CHOCOLATE CHIP COOKIES, CHOCOLATE CHIP, KEY F..."


In [33]:
display(rare10[['ingredient_norm', 'food_count', 'examples']])

,ingredient_norm,food_count,examples
0,100% organic young coconut water,1,"[HEALEO, COCONUT WATER COLD PRESSED JUICE]"
1,#1 mustard seed,1,"[YELLOW MUSTARD, YELLOW]"
2,100% organic unrefined virgin coconut oil,1,[ORGANIC UNREFINED VIRGIN COCONUT OIL]
3,026212,1,"[RUSSELL STOVER, FREEZE-IT CANDY BAR, VANILLA ..."
4,2% or less of calcium sulfate,1,[LAVASH ROLL-UPS]
5,100% organic durum wheat semolina,1,"[ORGANIC LINGUINI, ENRICHED MACARONI PRODUCT]"
6,100% new zealand honey,1,"[NATIVE FLOWER HONEY, NATIVE FLOWER]"
7,100% organic sesame seed protein,1,"[ORGANIC SESAME FLOUR, SESAME FLOUR]"
8,100% pure florida squeezed orange juice,1,[NO PULP PREMIUM ORANGE JUICE]
9,100% pure beef,1,"[ORIGINAL STEAKHOUSE BURGERS, ORIGINAL]"
